# Klygo — Reproducible NVIDIA L4 Correctness & Performance Evaluation (`detection-evaluation`)

This notebook runs the end-to-end Klygo benchmark and correctness evaluation on **Google Colab NVIDIA L4**:
1. Clones `https://github.com/IchigoMazone/klygo` on branch `detection-evaluation` into a clean Colab checkout.
2. Records hardware/software environment and verifies `nvidia-smi` / `is_nvidia_l4`.
3. Runs the full `unittest` suite and `audit_public_api.py --all`.
4. Runs `benchmarks/gpu_suite/evaluate_release.py --include-models --repeats 7` covering:
   - `klygo.metrics` (`fp16`, `fp32`, `fp64`, `packed=False/True`, `cpu/cuda`, `evaluate` vs `compose`)
   - `DetectionGroup` & `postprocessing` (`cpu/cuda`, `fp16/fp32`)
   - `datasets.load`, `media.save_images(size=...)`, `visual.compare_images`, and `utils.tabulate`
   - Multi-backend models (`yolo`, `yolo-world`, `faster-rcnn`, `grounding-dino`, `d-fine`) across `float32` and `float16` and all 6 execution modes.
5. Displays all summary tables using `klygo.utils.tabulate`.

In [ ]:
!nvidia-smi
%cd /content
!rm -rf klygo
!git clone --branch detection-evaluation --depth 1 https://github.com/IchigoMazone/klygo.git
%cd /content/klygo
!git rev-parse HEAD

In [ ]:
!pip install -q -e .
!pip install -q ultralytics transformers keras keras-hub cupy-cuda12x opencv-python-headless

In [ ]:
!python -m unittest discover -s test -t . -p "test*.py"
!python .agents/skills/klygo-module-guides/scripts/audit_public_api.py --all

In [ ]:
!python benchmarks/gpu_suite/evaluate_release.py \
  --output artifacts/benchmarks/colab_l4_eval \
  --repeats 7 \
  --include-models \
  --models yolo,yolo-world,faster-rcnn,grounding-dino,d-fine \
  --dtypes float32,float16 \
  --include-scaling \
  --scaling-models yolo,yolo-world

In [ ]:
import glob
from klygo import files, utils

report = files.load("artifacts/benchmarks/colab_l4_eval/summary.json", verbose=False)
print("=== Environment Summary ===")
utils.tabulate({k: v for k, v in report["environment"].items() if k not in ("git", "versions")}, title="Colab Runtime")

for section in ("metrics", "group_post", "io_visual", "models", "scaling"):
    rows = report.get(section, [])
    if rows:
        print(f"\n=== {section} ===")
        utils.tabulate(rows, title=section)

mode_rows = []
for path in sorted(glob.glob("artifacts/benchmarks/colab_l4_eval/inference/*/*.json")):
    data = files.load(path, verbose=False)
    for case in data.get("cases", []):
        mode_rows.append({
            "model": data["model"],
            "dtype": data["settings"]["dtype"],
            "actual_dtype": data.get("actual_model_dtype", ""),
            "mode": case["mode"],
            "fps_median": round(case["fps_median"], 2),
            "p50_ms": round(case["ready_interval_p50_ms"], 2),
            "parity_pass": case["all_parity_pass"],
        })
if mode_rows:
    print("\n=== 6-Mode Model Inference Details ===")
    utils.tabulate(mode_rows, title="NVIDIA L4 6-Mode Results")